Quiz 60 — Complete Distribution & Outlier Analysis  [SOLVED]
Theme  : HR Dashboard
Difficulty: Hard  |  CAPSTONE

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(230)
depts     = ["Tech","Sales","Ops","Finance","HR"]
dept_mean = {"Tech":75000,"Sales":55000,"Ops":45000,"Finance":65000,"HR":42000}

department = np.random.choice(depts, 200)
salary     = np.array([np.random.normal(dept_mean[d], 15000) for d in department])
salary     = salary.clip(20000, 200000)
perf_score = np.random.uniform(1, 10, 200)
years_exp  = np.random.randint(1, 25, 200)

# ── 1. Bonus calculation ──────────────────────────────────────────────────────
bonus_pct    = np.where(perf_score>=8, 0.15, np.where(perf_score>=6, 0.08, 0.03))
actual_bonus = salary * bonus_pct
print(f"Total bonus pool: £{actual_bonus.sum():,.0f}")

# ── 2. Per-department report ──────────────────────────────────────────────────
print("\nDepartment Report:")
for d in depts:
    m    = department == d
    s    = salary[m]
    b    = actual_bonus[m]
    print(f"  {d:8s}: n={m.sum():2d}  mean=£{s.mean():,.0f}  "
          f"median=£{np.median(s):,.0f}  std=£{s.std():,.0f}  bonus=£{b.mean():,.0f}")

# ── 3. IQR outliers per dept ──────────────────────────────────────────────────
print("\nIQR Outliers per Department:")
for d in depts:
    m    = department == d
    s    = salary[m]
    q1, q3 = np.percentile(s, [25,75])
    iqr  = q3 - q1
    out  = ((s < q1-1.5*iqr) | (s > q3+1.5*iqr)).sum()
    print(f"  {d:8s}: {out} outliers")

# ── 4. Correlation: years_exp vs salary ──────────────────────────────────────
corr = np.corrcoef(years_exp, salary)[0, 1]
print(f"\nCorrelation (years_exp vs salary): r={corr:.3f}")
if abs(corr) > 0.5:
    print("  → Strong correlation")
elif abs(corr) > 0.25:
    print("  → Moderate correlation")
else:
    print("  → Weak/no linear correlation")

# ── 5. Quadrant analysis ──────────────────────────────────────────────────────
med_sal     = np.median(salary)
high_perf   = perf_score >= 7
high_sal    = salary > med_sal
quadrants   = {
    "Stars"      : (high_perf &  high_sal).sum(),
    "Undervalued": (high_perf & ~high_sal).sum(),
    "Overpaid"   : (~high_perf &  high_sal).sum(),
    "At-risk"    : (~high_perf & ~high_sal).sum(),
}
print("\nPerformance-Salary Quadrant:")
for label, count in quadrants.items():
    print(f"  {label:12s}: {count}")

# ── 6. Scatter plot ───────────────────────────────────────────────────────────
dept_colours = {"Tech":"steelblue","Sales":"tomato","Ops":"green",
                "Finance":"purple","HR":"orange"}
colours_arr  = [dept_colours[d] for d in department]

fig, ax = plt.subplots(figsize=(11, 7))
for d, c in dept_colours.items():
    m = department == d
    ax.scatter(perf_score[m], salary[m]/1000, c=c, alpha=0.6, label=d, s=40)
ax.axvline(7, color="navy", linestyle="--", linewidth=1.5, label="Perf=7")
ax.axhline(med_sal/1000, color="black", linestyle="--", linewidth=1.5,
           label=f"Median £{med_sal/1000:.0f}k")
ax.set_xlabel("Performance Score")
ax.set_ylabel("Salary (£k)")
ax.set_title("HR Dashboard: Salary vs Performance by Department")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig("hard_20_hr_dashboard.png", dpi=100)
plt.show()

# ── WHY ───────────────────────────────────────────────────────────────────────
# np.corrcoef returns a 2×2 correlation matrix; [0,1] is the off-diagonal r value.
# r close to +1 = strong positive linear relationship.
# The quadrant analysis uses compound boolean masks — a pattern you'll use daily
# in Pandas with .loc[] boolean filtering.
# This capstone question combines every major NumPy concept from the quiz pack.